# Resume Scorer — Interactive Test

Use this notebook to test the Resume Scorer agent with different job URLs and resumes.

Every run is **automatically traced in LangSmith** — check your project dashboard to inspect
node-level inputs, outputs, and latency.

---
**Workflow:** `scrape_job` → `run_scorer`

| Node | What it does |
|---|---|
| `scrape_job` | Fetches and cleans the job description from the URL |
| `run_scorer` | Calls Gemini 2.5 Flash and returns a structured JSON assessment |

## Setup

Loads environment variables (`.env`) and imports the graph.
Run this cell once before anything else.

In [1]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# Add src/ to path so the agent package is importable when running outside Docker
sys.path.insert(0, str(Path("../src").resolve()))

# Load API keys from Agent/.env
load_dotenv("../.env")

from agent.graph import graph

api_key = os.getenv("GOOGLE_API_KEY", "")
print(f"GOOGLE_API_KEY : {api_key[:8]}{'*' * max(0, len(api_key) - 8)}")
print("Graph loaded. READY TO RUN BABY.")

GOOGLE_API_KEY : AIzaSyCl*******************************
Graph loaded. READY TO RUN BABY.


---
## 1. Define Your Inputs

Replace `JOB_URL` with a real posting and paste the resume text below.

In [2]:
JOB_URL = "https://www.linkedin.com/jobs/search-results/?currentJobId=4387414835&eBP=NOT_ELIGIBLE_FOR_CHARGING&refId=Bht0Do%2B56aT1mN1jyac22w%3D%3D&trackingId=Q%2BbGweksTJ1BD5ox6x6WVw%3D%3D&keywords=data%20scientist&origin=SEMANTIC_SEARCH_HISTORY&geoId=101165590&distance=25"

RESUME_TEXT = """
Caiua Utida  |  utida@gmail.com.com  |  linkedin.com/in/utida

EXPERIENCE
AI Engineer — Padrinhos Magicos Corp  (2021 – Present)
  • Built LLMs workflows with FastAPI and LangChain serving 50 000 users/day.
  • Led integration from scratch to end-to-end using Docker.
  • Reduced deployment time by 40 % with a GitHub Actions CI/CD pipeline.

Data Scientist Junior — Alpha Inc  (2019 – 2022)
  • Developed internal dashboards using Python.
  • Developed Deep Learning models for customer segmentation, improving targeting by 25 %.

SKILLS
Python, FastAPI, LangChain, Deep Learning, Docker, GitHub Actions, Data Visualization

EDUCATION
B.Sc. Computer Science — Super Choque University  (2019)
"""

---
## 2. Run the Agent

This will scrape the job URL and call Gemini 2.5 Flash.
The full trace will appear in your LangSmith dashboard under the `resume-scorer` project.

In [3]:
result = await graph.ainvoke({
    "job_url": JOB_URL,
    "resume_text": RESUME_TEXT,
})

score = result["score_result"]
print(json.dumps(score, indent=2))

Primary model rate-limited (429) — switching to fallback immediately.


{
  "match_score": 72,
  "matched_skills": [
    "Python",
    "Data Visualization",
    "Deep Learning",
    "Computer Science Degree",
    "GitHub Actions"
  ],
  "missing_skills": [
    "SQL",
    "A/B testing",
    "Cloud Data Platforms (BigQuery/Snowflake/Databricks)",
    "Marketing Analytics",
    "Statistical modeling (pandas/scikit-learn/statsmodels)"
  ],
  "experience_gap": "Candidate has ~5 years of experience, meeting the 2+ years requirement comfortably.",
  "top_improvements": [
    {
      "action": "Highlight SQL expertise and experience with cloud data warehouses (e.g., BigQuery, Snowflake).",
      "reason": "SQL and cloud platforms are essential for querying e-commerce data and are notably absent from the resume.",
      "priority": "high"
    },
    {
      "action": "Add specific examples of statistical experimentation (A/B testing) and business impact metrics.",
      "reason": "The JD emphasizes A/B testing and translating data science into commercial outcomes/g

---
## 3. Quick Summary

In [4]:
print(f"Match score : {score['match_score']}/100")
print(f"Summary     : {score['summary']}")
print()
print("Matched skills  :", ", ".join(score["matched_skills"]))
print("Missing skills  :", ", ".join(score["missing_skills"]))
print("Experience gap  :", score["experience_gap"])
print()
print("Top improvements:")
for i, item in enumerate(score["top_improvements"], 1):
    print(f"  {i}. [{item['priority'].upper()}] {item['action']}")
    print(f"     → {item['reason']}")

Match score : 72/100
Summary     : The candidate has strong technical foundations in AI/ML and deployment, meeting the seniority and educational requirements for the role. The biggest gaps are a lack of explicit mention of essential SQL skills, cloud data warehousing experience, and a focus on commercial business metrics like A/B testing.

Matched skills  : Python, Data Visualization, Deep Learning, Computer Science Degree, GitHub Actions
Missing skills  : SQL, A/B testing, Cloud Data Platforms (BigQuery/Snowflake/Databricks), Marketing Analytics, Statistical modeling (pandas/scikit-learn/statsmodels)
Experience gap  : Candidate has ~5 years of experience, meeting the 2+ years requirement comfortably.

Top improvements:
  1. [HIGH] Highlight SQL expertise and experience with cloud data warehouses (e.g., BigQuery, Snowflake).
     → SQL and cloud platforms are essential for querying e-commerce data and are notably absent from the resume.
  2. [HIGH] Add specific examples of statistical 